# Transcrição ASR — Original vs Trimmed

Notebook para comparar transcrição do áudio original e do áudio trimmed gerado pelo VAD/LSTM.

Entradas-alvo:
- Original: `audio/eval/S01_U06.CH1.wav`
- Trimmed: `results/clean_audio/S01_U06.CH1_trimmed.wav`
- CSV de referência temporal: `chime6_kinect_eval.csv`

In [ ]:
from pathlib import Path
import json
import pandas as pd

from transcription import (
    build_transcriber,
    transcribe_range,
    save_transcription_json,
 )

OUTPUT_DIR = Path("results/transcriptions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Baseline rápido para validar pipeline.
MODEL_ID = "openai/whisper-medium.en"
LANGUAGE = "en"
OUTPUT_SUFFIX = f"-{MODEL_ID}".replace("/", "_") + ".json"

# Chunk de inferência (segundos).
CHUNK_SEC = 30
BATCH_SIZE = 8
CHUNK_BATCH_SIZE = 8

## 1. Definir entradas (original, trimmed e CSV)

In [ ]:


ORIGINAL_WAV = Path("audio/eval/S01_U06.CH1.wav")

trimmed_candidates = [
    Path("results/clean_audio/S01_U06.CH1_trimmed.wav"),
    Path("results/clean_audio/S01_U06.CH1.wav"),
]
TRIMMED_WAV = next((p for p in trimmed_candidates if p.exists()), trimmed_candidates[0])

CSV_PATH = Path("chime6_kinect_eval.csv")
SAMPLE_PREFIX = "S01_U06"

if not ORIGINAL_WAV.exists():
    raise FileNotFoundError(f"Áudio original não encontrado: {ORIGINAL_WAV}")
if not TRIMMED_WAV.exists():
    raise FileNotFoundError(
        f"Áudio trimmed não encontrado. Tentados: {[str(p) for p in trimmed_candidates]}"
    )
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV não encontrado: {CSV_PATH}")

df_csv = pd.read_csv(CSV_PATH, usecols=["sample_id", "timestamp_ms"])
n_rows_prefix = int(df_csv["sample_id"].astype(str).str.startswith(SAMPLE_PREFIX).sum())

print("Original:", ORIGINAL_WAV)
print("Trimmed :", TRIMMED_WAV)
print("CSV     :", CSV_PATH)
print("Frames CSV para prefixo", SAMPLE_PREFIX + ":", n_rows_prefix)

Original: audio\eval\S01_U06.CH1.wav
Trimmed : results\clean_audio\S01_U06.CH1_trimmed.wav
CSV     : chime6_kinect_eval.csv
Frames CSV para prefixo S01_U06: 1192818


In [7]:
## Saídas JSON
filename_original_full = "S01_U06_original_full" + OUTPUT_SUFFIX
filename_trimmed_full = "S01_U06_trimmed_full" + OUTPUT_SUFFIX

## 2. Inicializar transcritor

In [3]:
asr, generate_kwargs = build_transcriber(model_id=MODEL_ID, language=LANGUAGE)
print("Model loaded:", MODEL_ID)
print("Generate kwargs:", generate_kwargs)

Loading weights: 100%|██████████| 947/947 [00:19<00:00, 48.54it/s]


Model loaded: openai/whisper-medium.en
Generate kwargs: {}


## 3. Smoke test (90s) — original vs trimmed

Teste curto para validar pipeline e comparar rapidamente a qualidade textual.

In [ ]:
smoke_original = transcribe_range(
    asr=asr,
    generate_kwargs=generate_kwargs,
    wav_path=ORIGINAL_WAV,
    start_sec=0.0,
    end_sec=90.0,
    chunk_sec=CHUNK_SEC,
    batch_size=BATCH_SIZE,
    chunk_batch_size=CHUNK_BATCH_SIZE,
)

smoke_trimmed = transcribe_range(
    asr=asr,
    generate_kwargs=generate_kwargs,
    wav_path=TRIMMED_WAV,
    start_sec=0.0,
    end_sec=90.0,
    chunk_sec=CHUNK_SEC,
    batch_size=BATCH_SIZE,
    chunk_batch_size=CHUNK_BATCH_SIZE,
)

print("Texto original (início):")
print(smoke_original.text[:800])
print("\n" + "=" * 80 + "\n")
print("Texto trimmed (início):")
print(smoke_trimmed.text[:800])

[transformers] Whisper did not predict an ending timestamp, which can happen if audio is cut off in the middle of a word. Also make sure WhisperTimeStampLogitsProcessor was used during generation.


Texto (inicio):
P05. Vancouver Stanley Park is 10% larger than Central Park, New York. One of the most powerful, popular ways to explore it is during a bike ride or walk along famous sea wall. Hi, I'm participant P06. Vancouver Stanley Park is 10% larger than Central Park, New York. one of the most popular r- ways to explore it is during a bike ride or walk along a famous seawall. Hi, I'm participant P07. Thank you for standing by the 10% margin in Central Park, New York. One of the most popular ways to explore it is during a bike ride or walk along a famous seawall. Hi, I'm participant P08, Vancouver Stanley Park, it's 10% more... One of the most popular ways to explore it is during bike ride or along the famous sea wall. Okay, let's see lunch. Okay, here's the pie. Can I help with anything? Yeah, that looks good. Hey, did you put the oven eating or did you put it out? No, I left it on it's pretty


In [ ]:
filename_original_smoke = "S01_U06_original_smoke_0_90s" + OUTPUT_SUFFIX
filename_trimmed_smoke = "S01_U06_trimmed_smoke_0_90s" + OUTPUT_SUFFIX

save_transcription_json(smoke_original, OUTPUT_DIR / filename_original_smoke)
save_transcription_json(smoke_trimmed, OUTPUT_DIR / filename_trimmed_smoke)

print("Salvo em", OUTPUT_DIR / filename_original_smoke)
print("Salvo em", OUTPUT_DIR / filename_trimmed_smoke)

Salvo em results\transcriptions\S02_smoke_0_90s-openai_whisper-small.en.json


## 4. Transcrever áudio original completo

In [ ]:
original_full = transcribe_range(
    asr=asr,
    generate_kwargs=generate_kwargs,
    wav_path=ORIGINAL_WAV,
    start_sec=0.0,
    end_sec=None,
    chunk_sec=CHUNK_SEC,
    batch_size=BATCH_SIZE,
    chunk_batch_size=CHUNK_BATCH_SIZE,
)

In [ ]:
save_transcription_json(original_full, OUTPUT_DIR / filename_original_full)
print("Original full salvo em", OUTPUT_DIR / filename_original_full)

Original full salvo em results\transcriptions\S01_U06_original_full-openai_whisper-medium.en.json


## 5. Transcrever áudio trimmed completo

In [4]:
trimmed_full = transcribe_range(
    asr=asr,
    generate_kwargs=generate_kwargs,
    wav_path=TRIMMED_WAV,
    start_sec=0.0,
    end_sec=None,
    chunk_sec=CHUNK_SEC,
    batch_size=BATCH_SIZE,
    chunk_batch_size=CHUNK_BATCH_SIZE,
)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precede

In [ ]:

save_transcription_json(trimmed_full, OUTPUT_DIR / filename_trimmed_full)
print("Trimmed full salvo em", OUTPUT_DIR / filename_trimmed_full)

Trimmed full salvo em results\transcriptions\S01_U06_trimmed_full-openai_whisper-medium.en.json


In [ ]:
print("Prévia original:")
print(original_full.text[:800])
print("\n" + "=" * 80 + "\n")
print("Prévia trimmed:")
print(trimmed_full.text[:800])

In [ ]:
len_original_chars = len(original_full.text)
len_trimmed_chars = len(trimmed_full.text)

print("Tamanho texto original (chars):", len_original_chars)
print("Tamanho texto trimmed  (chars):", len_trimmed_chars)
if len_original_chars > 0:
    print("Razão trimmed/original:", round(len_trimmed_chars / len_original_chars, 4))

In [ ]:
comparison_summary = pd.DataFrame([
    {"variant": "original", "json": filename_original_full, "text_chars": len(original_full.text)},
    {"variant": "trimmed",  "json": filename_trimmed_full,  "text_chars": len(trimmed_full.text)},
])
display(comparison_summary)

## 6. Inspecionar JSON salvo

In [ ]:
sample_json = OUTPUT_DIR / filename_trimmed_smoke
if sample_json.exists():
    with open(sample_json, encoding="utf-8") as f:
        data = json.load(f)
    print("arquivo:", sample_json.name)
    print("n_segments:", len(data.get("segments", [])))
    print("text preview:")
    print(data.get("text", "")[:1000])
else:
    print("Rode a célula de smoke test primeiro.")

n_segments: 15
text preview:
P05. Vancouver Stanley Park is 10% larger than Central Park, New York. One of the most powerful, popular ways to explore it is during a bike ride or walk along famous sea wall. Hi, I'm participant P06. Vancouver Stanley Park is 10% larger than Central Park, New York. one of the most popular r- ways to explore it is during a bike ride or walk along a famous seawall. Hi, I'm participant P07. Thank you for standing by the 10% margin in Central Park, New York. One of the most popular ways to explore it is during a bike ride or walk along a famous seawall. Hi, I'm participant P08, Vancouver Stanley Park, it's 10% more... One of the most popular ways to explore it is during bike ride or along the famous sea wall. Okay, let's see lunch. Okay, here's the pie. Can I help with anything? Yeah, that looks good. Hey, did you put the oven eating or did you put it out? No, I left it on it's pretty


## 7. Avaliação de erro de palavras (WER/CER)

Aqui comparamos a saída ASR com a transcrição de referência da sessão no mesmo intervalo de tempo.

Observação: nos áudios do CHiME6 há sobreposição de speakers. Esta avaliação concatena segmentos por tempo e funciona como baseline de comparação.

In [ ]:
from asr_metrics import evaluate_asr_output

REF_DIR = Path("official_transcriptions/eval")

# Para comparar original vs trimmed contra a mesma referência S01.
asr_eval_jobs = [
    # (OUTPUT_DIR / filename_original_smoke, REF_DIR / "S01.json"),
    # (OUTPUT_DIR / filename_trimmed_smoke, REF_DIR / "S01.json"),
    (OUTPUT_DIR / filename_original_full, REF_DIR / "S01.json"),
    (OUTPUT_DIR / filename_trimmed_full, REF_DIR / "S01.json"),
]

results = {}
for asr_json, ref_json in asr_eval_jobs:
    if not asr_json.exists():
        print(f"[PULADO] ASR não encontrado: {asr_json}")
        continue
    if not ref_json.exists():
        print(f"[PULADO] Referência não encontrada: {ref_json}")
        continue

    metrics = evaluate_asr_output(asr_json, ref_json)
    metrics["asr_json"] = asr_json.name
    metrics["ref_json"] = ref_json.name
    metrics["variant"] = "trimmed" if "trimmed" in asr_json.name else "original"
    results[asr_json.name] = metrics

if results:
    df_asr_metrics = pd.DataFrame(results.values())
    cols = [
        "variant", "asr_json", "ref_json", "duration_sec",
        "wer", "cer", "ref_words", "hyp_words", "ref_chars", "hyp_chars",
    ]
    display(df_asr_metrics[cols].round(4))
else:
    print("Nenhuma avaliação executada.")

,variant,asr_json,ref_json,duration_sec,wer,cer,ref_words,hyp_words,ref_chars,hyp_chars
0,original,S01_U06_original_full-openai_whisper-medium.en...,S01.json,9546.976,0.5747,0.4834,28004.0,23990.0,105563.0,92788.0
1,trimmed,S01_U06_trimmed_full-openai_whisper-medium.en....,S01.json,6665.408,0.7874,0.6701,20410.0,17266.0,77273.0,65301.0


In [ ]:
## Salvar metricas de erro como csv dividido em original_full e trimmed_full
METRICS_CSV = OUTPUT_DIR / f"asr_metrics_comparison{OUTPUT_SUFFIX}.csv"
df_asr_metrics.to_csv(METRICS_CSV, index=False)
print("Métricas salvas em", METRICS_CSV)